In [3]:
import requests
import re
import sys
import time
import json
import hashlib
import random
import threading
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse, quote
from html import escape
from datetime import datetime
from collections import defaultdict

# ============================================
# 第一部分：配置与常量
# ============================================

class Config:
    """全局配置类"""
    VERSION = "2.0-Competition"
    TIMEOUT = 8
    MAX_CRAWL = 30
    DELAY = 0.3

    # CVSS评分参考
    SEVERITY_SCORE = {
        "CRITICAL": 9.0,
        "HIGH": 7.0,
        "MEDIUM": 5.0,
        "LOW": 3.0,
        "INFO": 1.0
    }

    # WAF特征指纹库
    WAF_FINGERPRINTS = {
        "CloudFlare": ["cf-ray", "__cfduid", "cloudflare"],
        "安全狗": ["safedog", "waf/2.0"],
        "阿里云WAF": ["aliyun", "yundun"],
        "ModSecurity": ["mod_security", "ModSecurity"],
        "360WAF": ["360wzb", "wangzhan.360"],
    }

    # User-Agent池
    USER_AGENTS = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    ]

# ============================================
# 第二部分：可视化输出类
# ============================================

class ConsoleUI:
    """美化终端输出 - 兼容Windows CMD"""

    # ANSI颜色代码
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BLUE = '\033[94m'
    PURPLE = '\033[95m'
    CYAN = '\033[96m'
    BOLD = '\033[1m'
    END = '\033[0m'

    @staticmethod
    def banner():
        """显示横幅"""
        print(f"""{ConsoleUI.CYAN}
    ╔══════════════════════════════════════════╗
    ║     LightScan Pro v{Config.VERSION}  ║
    ║   综合漏洞扫描系统    ║
    ╚══════════════════════════════════════════╝
    {ConsoleUI.END}""")

    @staticmethod
    def section(title):
        """显示章节标题"""
        print(f"\n{ConsoleUI.BOLD}{ConsoleUI.PURPLE}[{title}]{ConsoleUI.END}")
        print(f"{ConsoleUI.PURPLE}{'='*50}{ConsoleUI.END}")

    @staticmethod
    def success(msg):
        print(f"{ConsoleUI.GREEN}[✓] {msg}{ConsoleUI.END}")

    @staticmethod
    def error(msg):
        print(f"{ConsoleUI.RED}[✗] {msg}{ConsoleUI.END}")

    @staticmethod
    def warning(msg):
        print(f"{ConsoleUI.YELLOW}[!] {msg}{ConsoleUI.END}")

    @staticmethod
    def info(msg):
        print(f"{ConsoleUI.BLUE}[*] {msg}{ConsoleUI.END}")

    @staticmethod
    def found_vuln(vuln_type, detail, severity="MEDIUM"):
        """统一漏洞输出格式"""
        color = {
            "CRITICAL": ConsoleUI.RED,
            "HIGH": ConsoleUI.YELLOW,
            "MEDIUM": ConsoleUI.BLUE,
            "LOW": ConsoleUI.GREEN,
            "INFO": ConsoleUI.END
        }.get(severity, ConsoleUI.END)

        print(f"  {color}[{severity}] {vuln_type}{ConsoleUI.END}")
        print(f"         {detail}")

    @staticmethod
    def progress(current, total, msg=""):
        """显示进度条"""
        percent = int((current / total) * 50)
        bar = "█" * percent + "░" * (50 - percent)
        print(f"\r  [{bar}] {current}/{total} {msg}", end="", flush=True)
        if current >= total:
            print()

# ============================================
# 第三部分：HTTP引擎（修正Content-Type）
# ============================================
import urllib3
# 关闭SSL警告（如果跳过证书验证）
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class HTTPEngine:
    """HTTP请求引擎（增强版）"""

    def __init__(self, verify_ssl=False, debug=False):
        self.session = requests.Session()
        self.session.verify = verify_ssl  # 控制是否验证SSL证书
        self.debug = debug

        # 更完善的浏览器请求头
        self.session.headers.update({
            "User-Agent": random.choice(Config.USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
            "Accept-Encoding": "gzip, deflate, br",
            "Cache-Control": "max-age=0",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
        })

    def request(self, url, method="GET", data=None, headers=None, timeout=None):
        """统一请求"""
        try:
            timeout = timeout or Config.TIMEOUT
            merged_headers = {}
            if headers:
                merged_headers.update(headers)

            # POST请求自动设置Content-Type
            if method.upper() == "POST" and data:
                merged_headers.setdefault("Content-Type", "application/x-www-form-urlencoded")

            if method.upper() == "GET":
                resp = self.session.get(url, timeout=timeout, headers=merged_headers)
            elif method.upper() == "POST":
                resp = self.session.post(url, data=data, timeout=timeout, headers=merged_headers)
            elif method.upper() == "HEAD":
                resp = self.session.head(url, timeout=timeout, headers=merged_headers)
            else:
                return None

            return resp

        except requests.exceptions.SSLError as e:
            if self.debug:
                print(f"[!] SSL证书错误: {e}")
            return None
        except requests.exceptions.ConnectionError as e:
            if self.debug:
                print(f"[!] 连接错误: {e}")
            return None
        except requests.exceptions.Timeout:
            if self.debug:
                print(f"[!] 请求超时: {url}")
            return None
        except requests.exceptions.MissingSchema:
            if self.debug:
                print(f"[!] URL格式错误: {url}")
            return None
        except Exception as e:
            if self.debug:
                print(f"[!] 未知异常: {e}")
            return None

    def get(self, url, **kwargs):
        return self.request(url, method="GET", **kwargs)

    def post(self, url, **kwargs):
        return self.request(url, method="POST", **kwargs)

    def head(self, url, **kwargs):
        return self.request(url, method="HEAD", **kwargs)





# ============================================
    # 第四部分：WAF检测与绕过（修正空指针）
# ============================================
class WAFDetector:
    """增强版WAF检测 - 支持多种探测方式"""

    # 主动发送可能触发WAF的恶意请求，通过响应特征判断
    WAF_TEST_PAYLOADS = [
        ("/../../etc/passwd", "路径遍历探测"),
        ("?id=1' OR '1'='1", "SQL注入探测"),
        ("?q=<script>alert(1)</script>", "XSS探测"),
    ]

    @staticmethod
    def detect(response):
        """被动检测：从正常响应头/内容中识别WAF指纹"""
        if response is None:
            return ["无响应(可能被WAF阻断)"]

        detected = []
        try:
            headers_str = str(dict(response.headers)).lower()
        except:
            headers_str = ""
        try:
            cookies_str = str(response.cookies.get_dict()).lower()
        except:
            cookies_str = ""
        try:
            body_str = response.text[:5000].lower()
        except:
            body_str = ""

        all_text = headers_str + cookies_str + body_str
        for waf_name, fps in Config.WAF_FINGERPRINTS.items():
            for fp in fps:
                if fp.lower() in all_text:
                    detected.append(waf_name)
                    break
        return detected if detected else ["未检测到WAF"]

    @staticmethod
    def active_probe(http_engine, base_url):
        """主动探测：发送恶意Payload，观察响应变化"""
        results = []
        for payload_path, desc in WAFDetector.WAF_TEST_PAYLOADS:
            test_url = urljoin(base_url, payload_path)
            try:
                r = http_engine.get(test_url, timeout=5)
                if r is None:
                    results.append(f"{desc} -> 无响应(可能被丢弃)")
                elif r.status_code == 403 or r.status_code == 406:
                    results.append(f"{desc} -> 被拦截(状态码{r.status_code})")
                else:
                    # 检查响应内容是否有WAF拦截特征
                    body = r.text[:1000].lower()
                    if any(w in body for w in ['waf', 'blocked', 'forbidden', 'access denied']):
                        results.append(f"{desc} -> 响应包含WAF拦截关键词")
            except:
                results.append(f"{desc} -> 请求异常")
        return results


# ============================================
    # 第五部分：漏洞检测模块（修正URL构造逻辑）
# ============================================
class SQLiDetector:
    """SQL注入检测器 - 减少误报，增强准确性"""

    def __init__(self, http_engine):
        self.http = http_engine
        # 参数值黑名单（不进行注入测试的参数名）
        self.skip_params = ['token', 'csrf', 'timestamp', 'nonce', 'sign', 'signature', 'captcha']

    def _build_url(self, base_url, params, target_param=None, payload=None):
        """正确构造测试URL，保留原有参数值"""
        if not params:
            return base_url

        query_parts = []
        for key, value in params.items():
            if target_param and key == target_param:
                # 将 payload 拼接在原始值后面（模拟注入）
                query_parts.append(f"{key}={value}{payload}")
            else:
                query_parts.append(f"{key}={value}")

        query_string = "&".join(query_parts)
        base = base_url.split('?')[0]
        return f"{base}?{query_string}"

    def _get_stable_baseline(self, url, method, params, target_param, retries=2):
        """
        获取稳定的基线响应长度。
        多次请求同一页面，若响应长度变化>5%则标记为不稳定页面，跳过检测。
        """
        lengths = []
        for _ in range(retries):
            if method.upper() == "GET":
                test_url = self._build_url(url, params, target_param, "")
                r = self.http.get(test_url)
            else:
                r = self.http.post(url, data=params)
            if r:
                lengths.append(len(r.text))
            time.sleep(0.1)

        if len(lengths) < 2:
            return None, False

        avg = sum(lengths) / len(lengths)
        # 波动超过5%认为页面动态性太强
        stable = all(abs(l - avg) / avg < 0.05 for l in lengths)
        return avg, stable

    def _response_length(self, url, method, params, target_param, payload):
        """发送一次测试请求并返回响应长度，失败返回None"""
        if method.upper() == "GET":
            test_url = self._build_url(url, params, target_param, payload)
            r = self.http.get(test_url)
        else:
            data = params.copy()
            data[target_param] = data[target_param] + payload
            r = self.http.post(url, data=data)
        return len(r.text) if r else None

    def detect(self, url, method="GET", params=None):
        """统一检测入口"""
        if not params:
            return []

        ConsoleUI.info(f"SQL注入检测: {url}")
        results = []

        for param in params:
            # 跳过黑名单参数
            if any(skip in param.lower() for skip in self.skip_params):
                continue

            # 首先进行报错注入（快速判断）
            error_result = self._error_based(url, method, params, param)
            if error_result:
                results.append(error_result)
                continue  # 已确认漏洞，跳过其他类型
             # 布尔盲注（带稳定性检查）
            blind_result = self._boolean_blind_enhanced(url, method, params, param)
            if blind_result:
                results.append(blind_result)
                continue

            # 时间盲注（作为兜底）
            time_result = self._time_based(url, method, params, param)
            if time_result:
                results.append(time_result)

        return results

    def _error_based(self, url, method, params, param):
        """基于报错注入"""
        error_payloads = ["'", '"', "')", "' OR '1'='1"]

        for payload in error_payloads:
            r = None
            if method.upper() == "GET":
                test_url = self._build_url(url, params, param, payload)
                r = self.http.get(test_url)
            else:
                data = params.copy()
                data[param] = data[param] + payload
                r = self.http.post(url, data=data)

            if not r:
                continue

            # 常见SQL错误特征
            error_patterns = [
                r"SQL syntax.*MySQL",
                r"Warning.*mysql_",
                r"Unclosed quotation mark",
                r"PostgreSQL.*ERROR",
                r"ORA-\d{5}",
                r"Microsoft OLE DB.*SQL",
                r"SQLite.*error",
                r"You have an error in your SQL",
            ]
            for pattern in error_patterns:
                if re.search(pattern, r.text, re.I):
                    return {
                        "type": "SQL注入(报错注入)",
                        "param": param,
                        "payload": payload,
                        "detail": f"参数 {param} 触发数据库错误信息泄露",
                        "severity": "HIGH"
                    }
        return None

    def _boolean_blind_enhanced(self, url, method, params, param):
        """增强型布尔盲注：稳定性检查 + 差异阈值动态调整"""
        # 获取基线稳定性
        avg_len, stable = self._get_stable_baseline(url, method, params, param)
        if not stable or avg_len is None:
            return None  # 页面太动态，不做盲注判断

        # 构造真假条件（拼接在原始值后）
        true_payload = "' AND 1=1 -- "
        false_payload = "' AND 1=2 -- "

        true_len = self._response_length(url, method, params, param, true_payload)
        if true_len is None:
            return None

        # 再次请求假条件前，重新获取一次真条件，防止网络波动
        true_len2 = self._response_length(url, method, params, param, true_payload)
        if true_len2 is None:
            return None
        true_len = min(true_len, true_len2)  # 取较小值减少误报

        false_len = self._response_length(url, method, params, param, false_payload)
        if false_len is None:
            return None

        # 差异阈值基于页面大小动态计算：取5%与80字节的较大值
        threshold = max(80, avg_len * 0.05)
        diff = abs(true_len - false_len)

        if diff > threshold:
            return {
                "type": "SQL注入(布尔盲注)",
                "param": param,
                "payload": true_payload,
                "detail": f"参数 {param} 布尔盲注差异 {diff} 字节 (阈值 {int(threshold)})",
                "severity": "HIGH"
            }
        return None

    def _time_based(self, url, method, params, param):
        """时间盲注（使用通用延时 payload）"""
        # 多种数据库的延时函数，取第一个能造成延时的
        time_payloads = [
            "' AND SLEEP(3)-- ",        # MySQL
            "' AND pg_sleep(3)-- ",     # PostgreSQL
            "' AND 1=DBMS_LOCK.SLEEP(3)-- ",  # Oracle
            "'; WAITFOR DELAY '0:0:3'-- ",    # MSSQL
        ]

        for payload in time_payloads[:2]:  # 只试前两个提高效率
            start = time.time()
            if method.upper() == "GET":
                test_url = self._build_url(url, params, param, payload)
                r = self.http.get(test_url, timeout=8)
            else:
                data = params.copy()
                data[param] = data[param] + payload
                r = self.http.post(url, data=data, timeout=8)
            elapsed = time.time() - start

            if elapsed > 3:
                return {
                    "type": "SQL注入(时间盲注)",
                    "param": param,
                    "payload": payload,
                    "detail": f"参数 {param} 触发延时 {elapsed:.1f}s",
                    "severity": "HIGH"
                }
        return None
    class XSSDetector:
      """XSS检测器 - 增强上下文感知，减少漏报"""

    def __init__(self, http_engine):
        self.http = http_engine
        # 分级Payload
        self.payloads = [
            # (payload, 危险上下文正则, 严重程度)
            ("<script>alert(1)</script>", r'<script[^>]*>.*?</script>', "HIGH"),
            ('"><script>alert(1)</script>', r'<script[^>]*>.*?</script>', "HIGH"),
            ("< img src=x onerror=alert(1)>", r' on\w+\s*=\s*["\']?', "MEDIUM"),
            ("<svg/onload=alert(1)>", r' on\w+\s*=\s*["\']?', "MEDIUM"),
        ]

    def _build_url(self, base_url, params, target_param, payload):
        """构造测试URL"""
        query_parts = []
        for key, value in params.items():
            if key == target_param:
                query_parts.append(f"{key}={payload}")
            else:
                query_parts.append(f"{key}={value}")
        query_string = "&".join(query_parts)
        base = base_url.split('?')[0]
        return f"{base}?{query_string}"

    def detect(self, url, method="GET", params=None):
        """XSS检测入口"""
        if not params:
            return []

        ConsoleUI.info(f"XSS检测: {url}")
        results = []

        for param in params:
            for payload, context_pattern, severity in self.payloads:
                r = None
                if method.upper() == "GET":
                    test_url = self._build_url(url, params, param, payload)
                    r = self.http.get(test_url)
                else:
                    data = params.copy()
                    data[param] = payload
                    r = self.http.post(url, data=data)

                if not r:
                    continue

                # 1. payload 完全原样出现在响应中
                if payload not in r.text:
                    continue

                # 2. 检查是否在危险上下文中
                #    寻找 payload 前后各200字符的区域
                index = r.text.find(payload)
                if index == -1:
                    continue
                start = max(0, index - 200)
                end = min(len(r.text), index + len(payload) + 200)
                context = r.text[start:end]

                # 如果周围有危险上下文（如 script 标签、事件处理器），则确认为漏洞
                if re.search(context_pattern, context, re.I):
                    results.append({
                        "type": f"XSS漏洞({severity})",
                        "param": param,
                        "payload": payload,
                        "detail": f"参数 {param} 存在反射型XSS，Payload位于可执行上下文",
                        "severity": severity
                    })
                    ConsoleUI.found_vuln("XSS", f"参数 {param} 反射型XSS", severity)
                    break  # 找到一个即停止该参数
        return results


class DirectoryScanner:
    """目录扫描器 - 增加软404检测，降低误报"""

    def __init__(self, http_engine):
        self.http = http_engine
        self.common_paths = [
            "admin", "login", "backup", "test", "api", "manage",
            "config", "debug", "wp-admin", "phpmyadmin",
            ".git/HEAD", "robots.txt", "sitemap.xml",
        ]
        # 软404特征：通常404页面会包含这些关键词
        self.soft_404_keywords = ['not found', '找不到', '不存在', '404', 'page not found']

    def _is_soft_404(self, response):
        """判断是否为软404（状态码200但内容是404）"""
        if not response:
            return False
        if response.status_code != 200:
            return False

        try:
            content = response.text[:500].lower()
        except:
            return False

        for keyword in self.soft_404_keywords:
            if keyword in content:
                return True
        return False

    def scan(self, base_url):
        """目录扫描"""
        ConsoleUI.section("目录与路径扫描")
        results = []
       # 获取基准404特征（访问随机路径）
        random_path = urljoin(base_url, f"nonexistent_{random.randint(10000,99999)}.html")
        base_404_response = self.http.get(random_path)
        
        for i, path in enumerate(self.common_paths):
            url = urljoin(base_url, path)
            try:
                r = self.http.head(url)
                if r is None:
                    r = self.http.get(url, timeout=4)
                
                if r is None:
                    continue
                
                status = r.status_code
                
                # 状态码200时检查软404
                if status == 200 and self._is_soft_404(r):
                    continue
                
                if status in [200, 301, 302, 403]:
                    # 如果状态码与基准404页面状态相同且内容相似，忽略
                    if base_404_response and status == base_404_response.status_code:
                        if base_404_response.status_code == 200:
                            # 简单比较长度（更准确可用相似度算法）
                            len_diff = abs(len(r.text) - len(base_404_response.text))
                            if len_diff < 50:  # 长度差异很小，大概率也是404
                                continue
                    
                    severity = "LOW"
                    if any(admin in path.lower() for admin in ["admin", "manage", "config", ".git"]):
                        severity = "MEDIUM"
                    if status == 200 and "admin" in path.lower():
                        severity = "HIGH"
                    
                    results.append({
                        "type": "敏感路径发现",
                        "url": url,
                        "detail": f"发现路径: {path} (状态码: {status})",
                        "severity": severity
                    })
                    ConsoleUI.found_vuln("敏感路径", f"{path} [{status}]", severity)
            except:
                pass
            time.sleep(Config.DELAY)
        
        return results


class HeaderAnalyzer:
    """HTTP头分析器 - 更新检查项，更贴近现代安全标准"""
    
    def __init__(self, http_engine):
        self.http = http_engine
    
    def analyze(self, url):
        """分析HTTP响应头"""
        ConsoleUI.section("HTTP安全头分析")
        results = []
        
        try:
            r = self.http.get(url)
        except:
            ConsoleUI.error("无法访问目标网站")
            return results
        
        if r is None:
            ConsoleUI.error("目标无响应")
            return results
        
        headers = r.headers
        
        # 现代化安全头检查
        checks = {
            "Content-Security-Policy": {
                "lacking": "缺少CSP头，存在XSS和数据注入风险",
                "severity": "MEDIUM",
                "extra": self._check_csp(headers.get("Content-Security-Policy", ""))
            },
            "Strict-Transport-Security": {
                "lacking": "缺少HSTS，存在SSL/TLS降级攻击风险",
                "severity": "MEDIUM"
            },
            "X-Content-Type-Options": {
                "lacking": "缺少X-Content-Type-Options，存在MIME嗅探风险",
                "severity": "LOW"
            },
            "X-Frame-Options": {
                "lacking": "缺少X-Frame-Options，可能遭受点击劫持 (CSP frame-ancestors 可替代)",
                "severity": "LOW"
            },
            "Referrer-Policy": {
                "lacking": "缺少Referrer-Policy，可能泄露URL信息",
                "severity": "INFO"
            }
        }
        
        for header, info in checks.items():
            if header not in headers:
                results.append({
                    "type": "HTTP安全头缺失",
                    "detail": info["lacking"],
                    "severity": info["severity"]
                })
                ConsoleUI.found_vuln("HTTP头", info["lacking"], info["severity"])
            elif "extra" in info and info["extra"]:
                results.append(info["extra"])
        
        # 信息泄露检查
        leak_headers = ["Server", "X-Powered-By", "X-AspNet-Version"]
        for h in leak_headers:
            if h in headers:
                results.append({
                    "type": "信息泄露",
                    "detail": f"响应头 {h}: {headers[h]}",
                    "severity": "INFO"
                })
                ConsoleUI.warning(f"信息泄露: {h}={headers[h]}")
        
        return results
    
    def _check_csp(self, csp_value):
        """检查CSP配置是否包含危险指令"""
        if not csp_value:
            return None
        dangers = []
        if 'unsafe-inline' in csp_value.lower():
            dangers.append("允许内联脚本/样式 (unsafe-inline)")
        if 'unsafe-eval' in csp_value.lower():
            dangers.append("允许eval() (unsafe-eval)")
        if dangers:
            return {
                "type": "CSP配置不当",
                "detail": "CSP存在风险指令: " + ", ".join(dangers),
                "severity": "MEDIUM"
            }
        return None 
    







# ============================================
    # 第六部分：爬虫（修正死循环）
# ============================================

class Crawler:
    """智能爬虫 - 修正死循环问题"""

    def __init__(self, http_engine):
        self.http = http_engine

    def crawl(self, base_url, max_pages=None):
        """爬取URL"""
        max_pages = max_pages or Config.MAX_CRAWL

        ConsoleUI.section("智能爬虫与参数收集")

        visited = set()
        to_visit = [base_url]
        urls_with_params = []

        # 解析基础URL用于同源检查
        base_domain = urlparse(base_url).netloc

        while to_visit and len(visited) < max_pages:
            url = to_visit.pop(0)

            # 跳过已访问的URL
            if url in visited:
                continue

            # 同源检查
            if urlparse(url).netloc and urlparse(url).netloc != base_domain:
                continue

            # 跳过静态资源
            static_extensions = ['.js', '.css', '.png', '.jpg', '.jpeg', '.gif', '.ico', '.svg', '.woff', '.ttf']
            if any(url.lower().endswith(ext) for ext in static_extensions):
                continue

            # 标记为已访问
            visited.add(url)
            ConsoleUI.info(f"爬取 [{len(visited)}/{max_pages}]: {url[:80]}")

            # 请求页面
            try:
                r = self.http.get(url, timeout=10)
            except:
                continue

            if r is None:
                continue

            # 检查是否成功获取内容
            if r.status_code != 200:
                continue

            # 如果URL包含参数，记录下来
            if '?' in url:
                parsed = urlparse(url)
                params = parse_qs(parsed.query)
                # 去重检查
                base = url.split('?')[0]
                if base not in [u['url'] for u in urls_with_params]:
                    urls_with_params.append({
                        "url": base,
                        "params": {k: v[0] for k, v in params.keys()},
                        "method": "GET"
                    })

            # 从页面提取链接
            try:
                content = r.text
            except:
                continue

            # 提取href链接
            links = re.findall(r'href=["\']([^"\']+)["\']', content, re.I)
            for link in links:
                # 去除锚点
                link = link.split('#')[0]

                # 跳过空链接和javascript
                if not link or link.startswith('javascript:') or link.startswith('mailto:'):
                    continue

                # 构造完整URL
                full_link = urljoin(url, link)

                # 避免重复加入
                if full_link not in visited and full_link not in to_visit:
                    # 限制队列大小
                    if len(to_visit) < max_pages * 2:
                        to_visit.append(full_link)

            # 礼貌延迟
            time.sleep(Config.DELAY)

        ConsoleUI.success(f"爬取完成: 发现 {len(urls_with_params)} 个带参数URL, 共爬取 {len(visited)} 页")
        return urls_with_params

# ============================================
    #第七部分：报告生成器
# ============================================

class ReportGenerator:
    """生成竞赛级HTML报告"""

    @staticmethod
    def generate(target, all_vulns, scan_info):
        """生成报告"""
        ConsoleUI.section("生成扫描报告")

        # 统计数据
        severity_count = defaultdict(int)
        type_count = defaultdict(int)

        for v in all_vulns:
            severity_count[v.get("severity", "INFO")] += 1
            type_count[v.get("type", "其他")] += 1

        # 计算总体风险评分
        total_severity = sum(
            Config.SEVERITY_SCORE.get(v.get("severity", "INFO"), 1.0)
            for v in all_vulns
        )
        risk_score = min(10, total_severity / 5) if all_vulns else 0

        if risk_score >= 8:
            risk_level = "严重"
            risk_color = "#e74c3c"
        elif risk_score >= 6:
            risk_level = "高"
            risk_color = "#e67e22"
        elif risk_score >= 3:
            risk_level = "中"
            risk_color = "#f39c12"
        else:
            risk_level = "低"
            risk_color = "#27ae60"

        # 生成HTML
        html = f"""<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <title>漏洞扫描报告 - {target}</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Microsoft YaHei', Arial, sans-serif; background: #f5f6fa; color: #2c3e50; line-height: 1.6; }}

        .header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 40px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        .header h1 {{ font-size: 32px; margin-bottom: 10px; }}
        .header .subtitle {{ opacity: 0.9; font-size: 16px; }}

        .container {{ max-width: 1200px; margin: 0 auto; padding: 30px 20px; }}

        .score-card {{
            background: white;
            border-radius: 15px;
            padding: 30px;
            margin-bottom: 30px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.08);
            text-align: center;
        }}
        .score {{ font-size: 80px; font-weight: bold; margin: 20px 0; }}

        .stats {{
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 20px;
            margin-bottom: 30px;
        }}
        .stat {{
            background: white;
            padding: 25px;
            border-radius: 10px;
            text-align: center;
            box-shadow: 0 2px 8px rgba(0,0,0,0.06);
        }}
        .stat .num {{ font-size: 42px; font-weight: bold; }}

        .section {{
            background: white;
            border-radius: 10px;
            padding: 25px;
            margin-bottom: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.06);
        }}
        .section-title {{
            font-size: 20px;
            font-weight: bold;
            padding-bottom: 15px;
            border-bottom: 2px solid #667eea;
            margin-bottom: 20px;
        }}

        .vuln-item {{
            border-left: 4px solid #e74c3c;
            padding: 12px 18px;
            margin: 10px 0;
            background: #fafafa;
            border-radius: 4px;
        }}
        .vuln-item.sev-high {{ border-left-color: #e74c3c; }}
        .vuln-item.sev-medium {{ border-left-color: #f39c12; }}
        .vuln-item.sev-low {{ border-left-color: #3498db; }}
        .vuln-type {{ font-weight: bold; font-size: 16px; }}
        .vuln-detail {{ color: #7f8c8d; margin-top: 5px; }}
        .vuln-payload {{
            background: #2c3e50;
            color: #ecf0f1;
            padding: 8px 12px;
            border-radius: 4px;
            font-family: 'Courier New', monospace;
            margin: 5px 0;
            word-break: break-all;
        }}

        .recommendation {{
            background: #eaf7ee;
            border-left: 4px solid #27ae60;
            padding: 15px;
            border-radius: 4px;
            margin: 10px 0;
        }}

        .footer {{
            text-align: center;
            padding: 30px;
            color: #95a5a6;
            font-size: 14px;
        }}

        @media print {{
            body {{ background: white; }}
            .header {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; }}
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🔍 LightScan Pro 漏洞扫描报告</h1>
        <div class="subtitle">
            <p><strong>扫描目标:</strong> {target}</p >
            <p><strong>扫描时间:</strong> {scan_info.get('start_time', 'N/A')}</p >
            <p><strong>扫描耗时:</strong> {scan_info.get('duration', 'N/A')}</p >
            <p><strong>扫描引擎:</strong> LightScan Pro v{Config.VERSION}</p >
        </div>
    </div>

    <div class="container">
        <!-- 风险评分 -->
        <div class="score-card">
            <h2>综合风险评分</h2>
            <div class="score" style="color: {risk_color}">{risk_score:.1f}<span style="font-size: 24px">/10</span></div>
            <p style="font-size: 20px; font-weight: bold; color: {risk_color}">风险等级: {risk_level}</p >
        </div>

        <!-- 统计卡片 -->
        <div class="stats">
            <div class="stat">
                <div class="num" style="color: #e74c3c">{severity_count.get('HIGH', 0) + severity_count.get('CRITICAL', 0)}</div>
                <div>高危漏洞</div>
            </div>
            <div class="stat">
                <div class="num" style="color: #f39c12">{severity_count.get('MEDIUM', 0)}</div>
                <div>中危漏洞</div>
            </div>
            <div class="stat">
                <div class="num" style="color: #3498db">{severity_count.get('LOW', 0)}</div>
                <div>低危漏洞</div>
            </div>
            <div class="stat">
                <div class="num">{len(all_vulns)}</div>
                <div>漏洞总计</div>
            </div>
        </div>

        <!-- 漏洞详情 -->
        <div class="section">
            <div class="section-title">📋 漏洞详情列表</div>
"""

        # 按严重程度排序
        severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3, "INFO": 4}
        sorted_vulns = sorted(all_vulns, key=lambda v: severity_order.get(v.get("severity", "INFO"), 99))

        for i, vuln in enumerate(sorted_vulns, 1):
            sev = vuln.get("severity", "MEDIUM")
            sev_class = "sev-high" if sev in ["CRITICAL", "HIGH"] else "sev-medium" if sev == "MEDIUM" else "sev-low"

            html += f"""
            <div class="vuln-item {sev_class}">
                <div class="vuln-type">[{sev}] #{i} {vuln.get('type', '未知漏洞')}</div>
                <div class="vuln-detail">
                    <strong>详情:</strong> {vuln.get('detail', '无')}<br>
                    <strong>位置:</strong> {vuln.get('param', vuln.get('url', 'N/A'))}<br>
"""
            if vuln.get('payload'):
                html += f'                    <div class="vuln-payload">Payload: {escape(str(vuln["payload"]))}</div>\n'

            html += """                </div>
            </div>
"""

        html += """
        </div>

        <!-- 修复建议 -->
        <div class="section">
            <div class="section-title">🔧 安全修复建议</div>
"""

        # 根据漏洞类型生成建议
        recommendations = set()
        for v in all_vulns:
            vtype = v.get("type", "")
            if "SQL注入" in vtype:
                recommendations.add("使用参数化查询（Prepared Statements）替代字符串拼接SQL")
                recommendations.add("对数据库用户实施最小权限原则")
            elif "XSS" in vtype:
                recommendations.add("对所有用户输入进行HTML实体编码（如使用htmlspecialchars）")
                recommendations.add("设置Content-Security-Policy头限制脚本执行")
            elif "HTTP" in vtype:
                recommendations.add("配置完整的安全响应头（CSP、HSTS、X-Frame-Options等）")
            elif "信息泄露" in vtype:
                recommendations.add("关闭服务器版本信息的显示（如Server、X-Powered-By头）")
            elif "路径" in vtype:
                recommendations.add("限制敏感目录的访问，使用.htaccess或Nginx配置禁止访问")

        for rec in list(recommendations)[:8]:
            html += f'            <div class="recommendation">✅ {rec}</div>\n'

        html += """
            <div class="recommendation">✅ 定期进行安全扫描和渗透测试，建立安全开发流程（SDLC）</div>
        </div>

        <div class="footer">
            <p>📊 本报告由 <strong>LightScan Pro</strong> 自动生成 | 仅供授权安全测试使用</p >
            <p>生成时间: """ + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + """</p >
        </div>
    </div>
</body>
</html>"""

        # 保存报告
        filename = f"scan_report_{int(time.time())}.html"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html)

        ConsoleUI.success(f"报告已生成: {filename}")
        return filename

# ============================================
 #第八部分：主扫描引擎
# ============================================

def XSSDetector(http):
    pass


class LightScanPro:
    """扫描主引擎"""

    def __init__(self, target, mode="full"):
        # 确保target有协议
        if not target.startswith(("http://", "https://")):
            target = "http://" + target

        self.target = target
        self.mode = mode
        self.http = HTTPEngine()
        self.vulns = []
        self.scan_info = {}

    def run(self):
        """执行扫描"""
        start_time = time.time()
        self.scan_info["start_time"] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        # 显示横幅
        ConsoleUI.banner()
        ConsoleUI.info(f"目标: {self.target}")
        ConsoleUI.info(f"模式: {self.mode}")
        print()

        # 第0步：WAF检测
        ConsoleUI.section("WAF检测")
        try:
            r = self.http.get(self.target)
            if r:
                wafs = WAFDetector.detect(r)
                ConsoleUI.warning(f"检测到WAF: {', '.join(wafs)}")
            else:
                ConsoleUI.error("目标无响应，请检查URL是否正确")
                return []
        except Exception as e:
            ConsoleUI.error(f"无法连接目标: {e}")
            return []

        # 第1步：爬虫收集URL
        crawler = Crawler(self.http)
        urls = crawler.crawl(self.target)

        if not urls:
            ConsoleUI.warning("未发现带参数的URL，使用基础页面检测")
            # 尝试从首页提取表单
            try:
                r = self.http.get(self.target)
                if r:
                    inputs = re.findall(r'<input[^>]*name=["\']([^"\']+)["\']', r.text, re.I)
                    if inputs:
                        params = {k: "test" for k in inputs}
                        urls = [{"url": self.target, "params": params, "method": "GET"}]
                    else:
                        urls = [{"url": self.target, "params": {"test": "1"}, "method": "GET"}]
                else:
                    urls = [{"url": self.target, "params": {"test": "1"}, "method": "GET"}]
            except:
                urls = [{"url": self.target, "params": {"test": "1"}, "method": "GET"}]

        # 第2步：HTTP头分析
        header_analyzer = HeaderAnalyzer(self.http)
        header_vulns = header_analyzer.analyze(self.target)
        self.vulns.extend(header_vulns)

        # 第3步：目录扫描
        dir_scanner = DirectoryScanner(self.http)
        dir_vulns = dir_scanner.scan(self.target)
        self.vulns.extend(dir_vulns)

        # 第4步：逐个URL核心检测
        sqli_detector = SQLiDetector(self.http)
        xss_detector = XSSDetector(self.http)

        total_urls = len(urls)
        ConsoleUI.section(f"核心漏洞检测 ({total_urls}个页面)")

        for i, url_info in enumerate(urls):
            ConsoleUI.progress(i+1, total_urls, f"检测中...")

            url = url_info["url"]
            method = url_info.get("method", "GET")
            params = url_info.get("params", {})

            if params:
                # SQL注入检测
                sqli_results = sqli_detector.detect(url, method, params)
                self.vulns.extend(sqli_results)

                # XSS检测
                xss_results = xss_detector.detect(url, method, params)
                self.vulns.extend(xss_results)

            time.sleep(Config.DELAY)

        ConsoleUI.progress(total_urls, total_urls, "完成!")
        print()

        # 第5步：生成报告
        scan_duration = f"{time.time() - start_time:.1f}秒"
        self.scan_info["duration"] = scan_duration

        report_file = ReportGenerator.generate(self.target, self.vulns, self.scan_info)

        # 输出总结
        ConsoleUI.section("扫描总结")
        ConsoleUI.success(f"扫描完成! 耗时: {scan_duration}")
        ConsoleUI.success(f"发现漏洞总数: {len(self.vulns)}")

        severity_count = defaultdict(int)
        for v in self.vulns:
            severity_count[v.get("severity", "INFO")] += 1

        print(f"""
    严重/高危: {severity_count.get('CRITICAL', 0) + severity_count.get('HIGH', 0)}
    中危: {severity_count.get('MEDIUM', 0)}
    低危: {severity_count.get('LOW', 0)}
    信息: {severity_count.get('INFO', 0)}

    详细报告: {report_file}
        """)

        return self.vulns

# ============================================
# 第九部分：程序入口
# ============================================
import sys
from your_scanner_module import , ConsoleUI   # 根据实际导入修改

def main():
    # 检查命令行参数
    if len(sys.argv) < 2:
        print("用法: python scanner.py <目标URL> [模式]")
        print("模式: quick (快速扫描) 或 full (完整扫描，默认)")
        sys.exit(1)

    target = sys.argv[1]
    mode = sys.argv[2] if len(sys.argv) > 2 else "full"

    try:
        scanner = (target, mode)
        scanner.run()
    except KeyboardInterrupt:
        print(f"\n{ConsoleUI.YELLOW}扫描被用户中断！{ConsoleUI.END}")
    except Exception as e:
        ConsoleUI.error(f"扫描异常：{e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()




    ╔══════════════════════════════════════════╗
    ║     LightScan Pro v2.0-Competition  ║
    ║   综合漏洞扫描系统    ║
    ╚══════════════════════════════════════════╝
    
[*] 目标: http://-f
[*] 模式: C:\Users\马兴艳\AppData\Local\Temp\kernel-a3d3750b-5b01-4c31-89a8-adbce9928caf16843755193151285726\connection.json


[WAF检测]
[✗] 目标无响应，请检查URL是否正确
